In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pickle, re, gzip
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

Mounted at /content/drive


In [5]:
SAVE_DIR = Path("/content/drive/MyDrive/semantic-drift/on_demand_contexts")
NEWS_RAW_DIR = Path("/content/drive/MyDrive/semantic-drift/news_crawl")
REDDIT_RAW_DIR = Path("/content/drive/MyDrive/semantic-drift/reddit_data/processed/monthly")

with open(SAVE_DIR / "top500_candidates.pkl", "rb") as f:
    top500_words = pickle.load(f)

top_words = top500_words[:100]

print("Loaded candidates:", len(top_words))
print(top_words[:30])

Loaded candidates: 100
['trashcan', 'bandaid', 'unsee', 'pansexual', 'fisting', 'spaceballs', 'bidets', 'suprised', 'westworld', 'crackhead', 'respawn', 'mansplaining', 'turds', 'cthulhu', 'gummies', 'neopets', 'cutscenes', 'dickheads', 'poopy', 'majora', 'rugrats', 'misophonia', 'deviantart', 'villian', 'nuking', 'ragnarok', 'whoring', 'sentience', 'kickass', 'mufasa']


In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
import json
import re
import gzip
from pathlib import Path

def extract_news_contexts(word, max_contexts=300):
    pattern = re.compile(rf"\b{re.escape(word)}\b", re.IGNORECASE)
    contexts = []

    for file in sorted(NEWS_RAW_DIR.glob("*.gz")):
        year_match = re.search(r"news\.(\d{4})", file.name)
        year = int(year_match.group(1)) if year_match else None

        with gzip.open(file, "rt", encoding="utf-8", errors="ignore") as f:
            for line in f:
                if pattern.search(line):
                    contexts.append({
                        "text": line.strip(),
                        "year": year
                    })

                    if len(contexts) >= max_contexts:
                        return contexts

    return contexts


def extract_reddit_contexts(word, max_contexts=300):
    pattern = re.compile(rf"\b{re.escape(word)}\b", re.IGNORECASE)
    contexts = []

    for file in sorted(REDDIT_RAW_DIR.rglob("*.jsonl")):
        year_match = re.search(r"(\d{4})-\d{2}", file.name)
        year = int(year_match.group(1)) if year_match else None

        with open(file, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    text = obj.get("body", "") or obj.get("selftext", "") or obj.get("title", "")
                except:
                    continue

                if pattern.search(text):
                    contexts.append({
                        "text": text.strip(),
                        "year": year
                    })

                    if len(contexts) >= max_contexts:
                        return contexts

    return contexts

In [5]:
def compute_drift(word, max_contexts=300, min_contexts=30):
    news_items = extract_news_contexts(word, max_contexts)
    reddit_items = extract_reddit_contexts(word, max_contexts)

    if len(news_items) < min_contexts or len(reddit_items) < min_contexts:
        return None

    news_texts = [x["text"] for x in news_items]
    reddit_texts = [x["text"] for x in reddit_items]

    news_emb = model.encode(news_texts, batch_size=64, normalize_embeddings=True)
    reddit_emb = model.encode(reddit_texts, batch_size=64, normalize_embeddings=True)

    news_centroid = np.mean(news_emb, axis=0)
    reddit_centroid = np.mean(reddit_emb, axis=0)

    sim = cosine_similarity(
        news_centroid.reshape(1, -1),
        reddit_centroid.reshape(1, -1)
    )[0][0]

    reddit_years = [x["year"] for x in reddit_items if x["year"] is not None]

    return {
        "word": word,
        "drift_score": float(1 - sim),
        "cosine_similarity": float(sim),
        "news_contexts": len(news_texts),
        "reddit_contexts": len(reddit_texts),
        "first_reddit_year": min(reddit_years) if reddit_years else None,
        "latest_reddit_year": max(reddit_years) if reddit_years else None,
        "old_examples": news_texts[:3],
        "new_examples": reddit_texts[:3]
    }

In [13]:
test_results = []

for word in top500_words[:3]:
    print("Testing:", word)
    res = compute_drift(word, max_contexts=100, min_contexts=10)
    test_results.append(res)

test_results

Testing: trashcan
Testing: bandaid
Testing: unsee


[{'word': 'trashcan',
  'drift_score': 0.15985184907913208,
  'cosine_similarity': 0.8401481509208679,
  'news_contexts': 100,
  'reddit_contexts': 100,
  'first_reddit_year': 2015,
  'latest_reddit_year': 2015,
  'old_examples': ['Another handbag survived the violent tugging of a mugger and several months lying waterlogged in a trashcan in Washington.',
   'The California Coastal Commission is asking all Californians to take responsibility for making sure trash goes where it belongs-securely in a trashcan, recycling bin, or a hazardous waste dump when appropriate.',
   'Gone are the unwanted video clips, the duplicated photos, the filed columns and the unlistened-to music, all consigned to the great Trashcan in the sky.'],
  'new_examples': ["Hit myself in the head with a trashcan as i was trying to pull the bag out of it. Unfortunately it wasn't my own trash since i work as a busboy",
   'I successfully threw my socks in my trashcan because I thought it was the laundry basket and puk

In [6]:
def build_contexts_fast(words, max_contexts=100):
    words = list(words)
    word_lookup = {w.lower(): w for w in words}

    pattern = re.compile(
        r"\b(" + "|".join(map(re.escape, sorted(words, key=len, reverse=True))) + r")\b",
        re.IGNORECASE
    )

    contexts = {w: {"news": [], "reddit": []} for w in words}

    for file in sorted(NEWS_RAW_DIR.glob("*.gz")):
        year_match = re.search(r"news\.(\d{4})", file.name)
        year = int(year_match.group(1)) if year_match else None
        print("News:", file.name)

        with gzip.open(file, "rt", encoding="utf-8", errors="ignore") as f:
            for line in f:
                for m in pattern.findall(line):
                    w = word_lookup.get(m.lower())
                    if w and len(contexts[w]["news"]) < max_contexts:
                        contexts[w]["news"].append({"text": line.strip(), "year": year})

    for file in sorted(REDDIT_RAW_DIR.rglob("*.jsonl")):
        year_match = re.search(r"(\d{4})-\d{2}", file.name)
        year = int(year_match.group(1)) if year_match else None
        print("Reddit:", file.name)

        with open(file, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    text = obj.get("body", "") or obj.get("selftext", "") or obj.get("title", "")
                except:
                    continue

                for m in pattern.findall(text):
                    w = word_lookup.get(m.lower())
                    if w and len(contexts[w]["reddit"]) < max_contexts:
                        contexts[w]["reddit"].append({"text": text.strip(), "year": year})

    return contexts

In [8]:
top_words = top500_words[:100]

print("Using words:", len(top_words))
print(top_words[:20])

Using words: 100
['trashcan', 'bandaid', 'unsee', 'pansexual', 'fisting', 'spaceballs', 'bidets', 'suprised', 'westworld', 'crackhead', 'respawn', 'mansplaining', 'turds', 'cthulhu', 'gummies', 'neopets', 'cutscenes', 'dickheads', 'poopy', 'majora']


In [9]:
contexts = build_contexts_fast(top_words, max_contexts=100)

with open(SAVE_DIR / "top100_contexts.pkl", "wb") as f:
    pickle.dump(contexts, f)

print("Saved contexts")

News: news.2007.en.shuffled.deduped.gz
News: news.2008.en.shuffled.deduped.gz
News: news.2009.en.shuffled.deduped.gz
News: news.2010.en.shuffled.deduped.gz
News: news.2011.en.shuffled.deduped.gz
News: news.2012.en.shuffled.deduped.gz
News: news.2013.en.shuffled.deduped.gz
News: news.2014.en.shuffled.deduped.gz
News: news.2015.en.shuffled.deduped.gz
Reddit: 2015-01.jsonl
Reddit: 2015-02.jsonl
Reddit: 2015-03.jsonl
Reddit: 2015-04.jsonl
Reddit: 2015-05.jsonl
Reddit: 2015-06.jsonl
Reddit: 2015-07.jsonl
Reddit: 2015-08.jsonl
Reddit: 2015-09.jsonl
Reddit: 2015-10.jsonl
Reddit: 2015-11.jsonl
Reddit: 2015-12.jsonl
Reddit: 2016-01.jsonl
Reddit: 2016-02.jsonl
Reddit: 2016-03.jsonl
Reddit: 2016-04.jsonl
Reddit: 2016-05.jsonl
Reddit: 2016-06.jsonl
Reddit: 2016-07.jsonl
Reddit: 2016-08.jsonl
Reddit: 2016-09.jsonl
Reddit: 2016-10.jsonl
Reddit: 2016-11.jsonl
Reddit: 2016-12.jsonl
Reddit: 2017-01.jsonl
Reddit: 2017-02.jsonl
Reddit: 2017-03.jsonl
Reddit: 2017-04.jsonl
Reddit: 2017-05.jsonl
Reddit: 201

In [7]:
with open(SAVE_DIR / "top100_contexts.pkl", "rb") as f:
    contexts = pickle.load(f)

print("Loaded contexts for:", len(contexts))
print(list(contexts.keys())[:10])

Loaded contexts for: 100
['trashcan', 'bandaid', 'unsee', 'pansexual', 'fisting', 'spaceballs', 'bidets', 'suprised', 'westworld', 'crackhead']


In [8]:
def compute_drift_from_contexts(word, contexts, min_contexts=20):
    news_items = contexts[word]["news"]
    reddit_items = contexts[word]["reddit"]

    if len(news_items) < min_contexts or len(reddit_items) < min_contexts:
        return None

    news_texts = [x["text"] for x in news_items]
    reddit_texts = [x["text"] for x in reddit_items]

    news_emb = model.encode(news_texts, batch_size=64, normalize_embeddings=True)
    reddit_emb = model.encode(reddit_texts, batch_size=64, normalize_embeddings=True)

    sim = cosine_similarity(
        np.mean(news_emb, axis=0).reshape(1, -1),
        np.mean(reddit_emb, axis=0).reshape(1, -1)
    )[0][0]

    reddit_years = [x["year"] for x in reddit_items if x["year"] is not None]

    return {
        "word": word,
        "drift_score": float(1 - sim),
        "cosine_similarity": float(sim),
        "news_contexts": len(news_texts),
        "reddit_contexts": len(reddit_texts),
        "first_reddit_year": min(reddit_years) if reddit_years else None,
        "latest_reddit_year": max(reddit_years) if reddit_years else None,
        "old_examples": news_texts[:3],
        "new_examples": reddit_texts[:3],
    }

In [9]:
results = []

for i, word in enumerate(contexts.keys(), 1):
    print(f"[{i}/{len(contexts)}] {word}")

    res = compute_drift_from_contexts(word, contexts)

    if res:
        results.append(res)

[1/100] trashcan
[2/100] bandaid
[3/100] unsee
[4/100] pansexual
[5/100] fisting
[6/100] spaceballs
[7/100] bidets
[8/100] suprised
[9/100] westworld
[10/100] crackhead
[11/100] respawn
[12/100] mansplaining
[13/100] turds
[14/100] cthulhu
[15/100] gummies
[16/100] neopets
[17/100] cutscenes
[18/100] dickheads
[19/100] poopy
[20/100] majora
[21/100] rugrats
[22/100] misophonia
[23/100] deviantart
[24/100] villian
[25/100] nuking
[26/100] ragnarok
[27/100] whoring
[28/100] sentience
[29/100] kickass
[30/100] mufasa
[31/100] telekinesis
[32/100] pantera
[33/100] grandpas
[34/100] whatcha
[35/100] everclear
[36/100] jumanji
[37/100] commies
[38/100] karama
[39/100] mulaney
[40/100] hitlers
[41/100] merica
[42/100] teleported
[43/100] trebuchet
[44/100] whataburger
[45/100] aragorn
[46/100] ejaculating
[47/100] roleplaying
[48/100] hearthstone
[49/100] embarassed
[50/100] sidenote
[51/100] abrahamic
[52/100] homeroom
[53/100] deftones
[54/100] obnoxiously
[55/100] waker
[56/100] roomie
[57

In [10]:
ranked_df = pd.DataFrame(results).sort_values("drift_score", ascending=False)

save_path = SAVE_DIR / "ranked_drift_top100.csv"
ranked_df.to_csv(save_path, index=False)

print("Saved:", save_path)
ranked_df.head(20)

Saved: /content/drive/MyDrive/semantic-drift/on_demand_contexts/ranked_drift_top100.csv


,word,drift_score,cosine_similarity,news_contexts,reddit_contexts,first_reddit_year,latest_reddit_year,old_examples,new_examples
74,sitewide,0.659313,0.340687,100,100,2015,2015,[How is this any different than a sitewide fil...,[All sides of an issue get accused of being a ...
40,merica,0.528275,0.471725,100,100,2015,2015,"[One recent program was devoted to wetlands, w...","[Merica, Cuz 'Merica!, Merica!]"
87,recieve,0.517527,0.482473,100,100,2015,2015,[They are accused of separate murders and coul...,[Wh did you recieve kill yourself messages? wt...
68,inbetween,0.495981,0.504019,100,100,2015,2015,[If you are older you have not only experience...,[Stop living off of your husband's money you l...
78,pecker,0.434871,0.565129,100,100,2015,2015,"[They flirt and swoon, but Sir Walter's inabil...","[The civil war started here, but mostly we're ..."
19,majora,0.431246,0.568754,100,100,2015,2015,[Among the charity creators: people like Major...,[Legend of Zelda: Ocarina of Time. \n\nToss up...
97,elden,0.405834,0.594166,100,100,2015,2018,"[Mayfield's attorneys -- Gerry Spence, Elden R...","[That fucking made me laugh, man. Also, you l..."
49,sidenote,0.387208,0.612792,100,100,2015,2015,"[""On a sidenote, we should recognize that lip ...",[>I was at a street festival and struck a conv...
26,whoring,0.369686,0.630314,100,100,2015,2015,"[There was an old man named Warren,/who hated ...","[Miley Cyrus and her attention whoring, Fuck t..."
56,catan,0.303824,0.696176,100,100,2015,2015,[David Brown in Praia da Luz and Tom Catan in ...,"[First get her hooked on heroin. Then, ""Oh, yo..."


In [11]:
def get_top_words(n=50):
    df = pd.read_csv(SAVE_DIR / "ranked_drift_top100.csv")
    return df.sort_values("drift_score", ascending=False).head(n)

get_top_words(10)

,word,drift_score,cosine_similarity,news_contexts,reddit_contexts,first_reddit_year,latest_reddit_year,old_examples,new_examples
0,sitewide,0.659313,0.340687,100,100,2015,2015,['How is this any different than a sitewide fi...,"[""All sides of an issue get accused of being a..."
1,merica,0.528275,0.471725,100,100,2015,2015,"['One recent program was devoted to wetlands, ...","['Merica', ""Cuz 'Merica!"", 'Merica!']"
2,recieve,0.517527,0.482473,100,100,2015,2015,['They are accused of separate murders and cou...,['Wh did you recieve kill yourself messages? w...
3,inbetween,0.495981,0.504019,100,100,2015,2015,"[""If you are older you have not only experienc...","[""Stop living off of your husband's money you ..."
4,pecker,0.434871,0.565129,100,100,2015,2015,"[""They flirt and swoon, but Sir Walter's inabi...","[""The civil war started here, but mostly we're..."
5,majora,0.431246,0.568754,100,100,2015,2015,['Among the charity creators: people like Majo...,"[""Legend of Zelda: Ocarina of Time. \n\nToss u..."
6,elden,0.405834,0.594166,100,100,2015,2018,"[""Mayfield's attorneys -- Gerry Spence, Elden ...","['That fucking made me laugh, man. Also, you ..."
7,sidenote,0.387208,0.612792,100,100,2015,2015,"['""On a sidenote, we should recognize that lip...",['>I was at a street festival and struck a con...
8,whoring,0.369686,0.630314,100,100,2015,2015,"['There was an old man named Warren,/who hated...","['Miley Cyrus and her attention whoring', 'Fuc..."
9,catan,0.303824,0.696176,100,100,2015,2015,['David Brown in Praia da Luz and Tom Catan in...,"['First get her hooked on heroin. Then, ""Oh, y..."
